In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Imports

In [2]:
import os
import sys

sys.path.append("..")

import random

import numpy as np
import torch
import torch.distributions as TD
from tqdm import tqdm

import wandb
from src.models.energy_based import EGEOT
from src.samplers.energy_based.sample_buffer import SampleBufferEgEOT
from src.samplers.from_dataset import DatasetSampler
from src.utils.train import compute_loss, update_average
from src.plotting.distributions import plot_images

In [3]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [4]:
torch.set_default_device(device)
# dtype = torch.float64
# torch.torch.set_default_dtype(dtype)

## 2. Dataset

In [5]:
from transformation_cr.data import get_training_set, get_test_set
from torch.utils.data import DataLoader

In [6]:
DATA_DIR = './datasets/BSD500_20percent/images'
DATA_DIR_WHOLE = './datasets/BSD500/images'
UPSCALE_FACTOR = 3
THREADS = 4

PAIRED_BATCH_SIZE = 4
UNPAIRED_BATCH_SIZE = 40
TEST_BATCH_SIZE = 100

In [7]:
train_set = get_training_set(DATA_DIR, UPSCALE_FACTOR)
training_data_loader = DataLoader(
    dataset=train_set,
    num_workers=THREADS,
    batch_size=PAIRED_BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator(device=device),
)

In [8]:
# Loading rest of the data that is fed only in the unsupervised TCR chain
train_set_whole = get_training_set(DATA_DIR_WHOLE, UPSCALE_FACTOR)
training_data_loader_un = DataLoader(
    dataset=train_set_whole,
    num_workers=THREADS,
    batch_size=UNPAIRED_BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator(device=device),
)  # The batch size for unsupervised data is more than supervised data

In [9]:
# Loading the Test Set for Evaluation
test_set = get_test_set(DATA_DIR, UPSCALE_FACTOR)
testing_data_loader = DataLoader(
    dataset=test_set,
    num_workers=THREADS,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    generator=torch.Generator(device=device),
)

In [10]:
# Examine data
data_sup = next(iter(training_data_loader))
input, target = data_sup[0].to(device), data_sup[1].to(device)
input.shape, target.shape

(torch.Size([4, 1, 85, 85]), torch.Size([4, 1, 255, 255]))

In [11]:
data_un = next(iter(training_data_loader_un))
input_un, target_un = data_un[0].to(device), data_un[1].to(device) 
input_un.shape, target_un.shape

(torch.Size([40, 1, 85, 85]), torch.Size([40, 1, 255, 255]))

## 2. Config

In [12]:
# from configs.energy_based.cost import MLPCostConfig, MLPLSECostConfig, MLPL2CostConfig
# from configs.energy_based.dataset import DatasetConfig
from configs.energy_based.model import EBMConfig
from configs.energy_based.optimizer import OptPairedConfig, OptUnpairedConfig
# from configs.energy_based.potential import PotentialConfig
from configs.energy_based.sampling import LangevinConfig
from configs.energy_based.train import TrainConfig

In [13]:
# Q_X_UNPAIRED_SAMPLES = 6990
# R_Y_UNPAIRED_SAMPLES = 7141
# P_XY_PAIRED_SAMPLES = 128
LR_PAIRED = 2e-4
LR_UNPAIRED = 2e-4
SAMPLING_NUM_ITER = 10
NUM_EPOCHS = 10
# PAIRED_BATCH_SIZE = 1024
# UNPAIRED_BATCH_SIZE = 1024

In [14]:
# NUM_LABELED = 10
# DATASET = 'mnist2usps'
# COST = 'Weak_Energy'
# DATASET_PATH = '../datasets/'

# BATCH_SIZE = 256
# C_SIZE = 1
# Z_SIZE = 2
# IMG_SIZE = 32
# T_ITERS = 10
# D_LR = 1e-5 
# T_LR = 1e-5
# NC = 1
# ZD = 128
# Z_STD = 1.
# PLOT_INTERVAL = 500
# CPKT_INTERVAL = 10000
# MAX_STEPS = 2501 #For illustration purpose only. For training use 60001
# SEED = 0x000001

In [15]:
# dataset_config = DatasetConfig(
#     P_XY_paired=P_XY_PAIRED_SAMPLES, Q_X_unpaired=Q_X_UNPAIRED_SAMPLES, R_Y_unpaired=R_Y_UNPAIRED_SAMPLES
# )
model_config = EBMConfig(sampling=LangevinConfig(num_iterations=SAMPLING_NUM_ITER))

opt_unpaired_config = OptUnpairedConfig(lr=LR_UNPAIRED)
opt_paired_config = OptPairedConfig(lr=LR_PAIRED)

train_config = TrainConfig(paired_batch_size=PAIRED_BATCH_SIZE, unpaired_batch_size=UNPAIRED_BATCH_SIZE)

In [16]:
# torch.manual_seed(train_config.seed)
# np.random.seed(train_config.seed)
# random.seed(train_config.seed)

## 4. Model initialization

In [17]:
from src.costs.tcr import TCRCost
from src.potentials.tcr import ResNet50Potential, ResNet18Potential
from src.potentials.convolutional import VanillaPotential

In [18]:
cost = TCRCost(UPSCALE_FACTOR, device)
# potential = ResNet18Potential([256, 256, 256],  lambda: torch.nn.LeakyReLU(0.2), train_resnet=False)
potential = VanillaPotential([256, 256, 256],  lambda: torch.nn.LeakyReLU(0.2), n_c=1, n_f=85)

In [19]:
# TODO: add to config
BASIC_NOISE_VAR = 1.0
P_SAMPLE_BUFFER_REPLAY = 0.95
SAMPLE_BUFFER_SAMPLES = 10000

In [20]:
basic_noise_gen = TD.Normal(
    torch.zeros_like(target_un[0]).to(device),
    torch.ones_like(target_un[0]).to(device) * BASIC_NOISE_VAR,
)

sample_buffer_instance = SampleBufferEgEOT(
    basic_noise_gen, p=P_SAMPLE_BUFFER_REPLAY, max_samples=SAMPLE_BUFFER_SAMPLES, device=device
)

In [21]:
model = EGEOT(potential, cost, sample_buffer_instance, model_config)

In [22]:
# For EMA update
if train_config.ema_update:
    model_copy = EGEOT(potential, cost, sample_buffer_instance, model_config)

## 5. Optimizers initialization

In [23]:
D_opt_unpaired = torch.optim.Adam(model.potential.parameters(), **opt_unpaired_config.model_dump())

In [24]:
D_opt_paired = torch.optim.Adam(model.cost.parameters(), **opt_paired_config.model_dump())

In [25]:
# TODO: refactor this config
EXP_NAME = (
    f"EgEOT_BSD500_"
    + f"LR_PAIRED_{opt_paired_config.lr}_"
    + f"LR_UNPAIRED_{opt_unpaired_config.lr}_"
    + f"SAMPLING_STEPS_{model_config.sampling.num_iterations}_"
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    D_LR_PAIRED=opt_paired_config.lr,
    D_LR_UNPAIRED=opt_unpaired_config.lr,
    BATCH_SIZE=train_config.unpaired_batch_size,
    # P_XY_PAIRED_SAMPLES=dataset_config.P_XY_paired,
    # Q_X_UNPAIRED_SAMPLES=dataset_config.Q_X_unpaired,
    # R_Y_UNPAIRED_SAMPLES=dataset_config.R_Y_unpaired,
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)

In [26]:
if train_config.steps_from > 0:
    D_opt_unpaired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_from}.pt")))
    D_opt_paired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_from}.pt")))

## 6. Model training

In [27]:
from math import log10

In [28]:
criterion_mse = torch.nn.MSELoss()

In [29]:
def test():
    avg_psnr = 0
    with torch.no_grad():
        for batch in testing_data_loader:
            input, target = batch[0].to(device), batch[1].to(device)
            
            prediction = model(input)
            mse = criterion_mse(prediction, target)
            psnr = 10 * log10(1 / mse.item())
            avg_psnr += psnr
    return avg_psnr / len(testing_data_loader)

In [30]:
model.compute_unpaired_loss(input_un, target_un)

{'loss': tensor(-0.0838, device='cuda:0', grad_fn=<AddBackward0>),
 'int_potential': tensor(-0.0441, device='cuda:0', grad_fn=<MeanBackward0>),
 'int_log_Z': tensor(-0.1278, device='cuda:0', grad_fn=<MeanBackward0>)}

In [31]:
wandb.init(name=EXP_NAME, config=config)

for step in range(NUM_EPOCHS):
    for iteration, batch in tqdm(enumerate(zip(training_data_loader, training_data_loader_un), 0)):
        cur_step = len(training_data_loader) * step + iteration
        data_sup, data_un = batch[0] , batch[1] #.to(device), batch[1].to(device)
        input, target = data_sup[0].to(device), data_sup[1].to(device)   # Here the data is used in supervised fashion
        input_un, target_un = data_un[0].to(device), data_un[1].to(device)   # Here the labels are not used

        # training loop
        D_opt_unpaired.zero_grad()

        output_unpaired = model.compute_unpaired_loss(input_un, target_un, compute_stats=True)
        D_loss_unpaired = output_unpaired["loss"]

        wandb.log({f"Unpaired: Loss": D_loss_unpaired.item()}, step=cur_step)
        wandb.log({f"Unpaired: \int f(y)": output_unpaired["int_potential"].item()}, step=cur_step)
        wandb.log({f"Unpaired: \int\log Z": output_unpaired["int_log_Z"].item()}, step=cur_step)
        wandb.log({f"Unpaired: -E(x, y)": output_unpaired["neg_energy_t"].item()}, step=cur_step)
        wandb.log({f"Unpaired: c(x, y)": output_unpaired["cost_t"].item()}, step=cur_step)
        wandb.log({f"Unpaired: f(y)": output_unpaired["potential_t"].item()}, step=cur_step)
        wandb.log({f"Unpaired: noise": output_unpaired["noise"].item()}, step=cur_step)

        D_opt_paired.zero_grad()
        output_paired = model.compute_paired_loss(input, target, compute_stats=True)
        D_loss_paired = output_paired["loss"]

        wandb.log({f"Paired: Loss": D_loss_paired.item()}, step=cur_step)
        # wandb.log({f"Paired: -E(x, y)": output_paired["neg_energy_t"].item()}, step=cur_step)
        # wandb.log({f"Paired: c(x, y)": output_paired["cost_t"].item()}, step=cur_step)
        # wandb.log({f"Paired: f(y)": output_paired["potential_t"].item()}, step=cur_step)
        # wandb.log({f"Paired: noise": output_paired["noise"].item()}, step=cur_step)

        D_loss = D_loss_unpaired + D_loss_paired
        D_loss.backward()
        D_opt_paired.step()
        D_opt_unpaired.step()

        if train_config.ema_update:
            update_average(model_copy, model, 0.99)
            model = model_copy
        else:
            model = model

        wandb.log({f"Loss": D_loss}, step=cur_step)
        

    torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"model_{cur_step}.pt"))
    wandb.log({f"Avg. PSNR, dB": test()}, step=cur_step)

torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f"D_{train_config.steps_to}.pt"))
torch.save(D_opt_paired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_to}.pt"))
torch.save(D_opt_unpaired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_to}.pt"))

wandb.finish()

wandb: Currently logged in as: muxaujl11110. Use `wandb login --relogin` to force relogin


10it [00:17,  1.78s/it]
10it [00:17,  1.76s/it]
10it [00:17,  1.76s/it]
10it [00:17,  1.77s/it]
10it [00:17,  1.78s/it]
10it [00:17,  1.79s/it]
10it [00:18,  1.81s/it]
10it [00:18,  1.81s/it]
10it [00:18,  1.82s/it]
10it [00:18,  1.83s/it]


Loss,██▁
Paired: Loss,▁▁▁█
"Unpaired: -E(x, y)",▁▁█
Unpaired: Loss,██▁
Unpaired: \int f(y),███▁
Unpaired: \int\log Z,██▁
"Unpaired: c(x, y)",▁▁█
Unpaired: f(y),▁▁▁█
Unpaired: noise,▅▇▄▅▅▅▄▃▆▅▃▅▄▃▆▅▆▁▆▅▆▃▇▆▅▆▇▅▄▇▄▄▃▄▇▄▆▅▇█
"Avg. PSNR, dB",nan
Loss,nan


## Plotting

In [ ]:
plot_swiss_roll(
    {"EBM": model},
    X_sampler,
    Y_sampler,
    X_paired_train,
    Y_paired_train,
    starting_points,
    gt_Y_points,
) 